In [1]:
import qewton

path = "/localdata/tomfre/SolutionData/MinimalSurface/"

X = qewton.Variable("x", 2)
U = qewton.Variable("u", 1) 
N = qewton.Variable("n", 2)

In [2]:
omega = qewton.geometries.MeshGeometry.load_mesh(
    variable=X, 
    file_path=path + "unit_square_minus_circle.msh",
    prune_z=True
)

gamma_D = omega.boundary.get_submesh("dirichlet")
gamma_N = omega.boundary.get_submesh("neumann")

/home/tomfre/Desktop/pioneer-backend/src/qewton/backends/torch/base.py:90: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:278.)
  return torch.as_tensor(data, dtype=converted_type, device=cls.get_device(device))


In [3]:
qewton.viz.Figure(qewton.viz.GeometryPlot(omega)).show()

In [4]:
inner_sampler = qewton.GridSampler(omega, n_points=25000)
gamma_D_sampler = qewton.GridSampler(gamma_D, n_points=8000)
gamma_N_sampler = qewton.GridSampler(
    gamma_N, n_points=8000, normal_name=N, compute_normals=True)

In [5]:
model = qewton.FCN(in_neurons=X, 
                   out_neurons=U, 
                   hidden_neurons=32, 
                   n_hidden_layers=3,
                   activation=qewton.bb.Tanh)

In [6]:
def pde_residual(u : U, x : X):
    u_grad = u.gradient(x)
    u_grad_norm = u_grad.norm()
    scaling = (1.0 + u_grad_norm**2).sqrt()
    return - (u_grad / scaling).div(x) + 1.0

pde_constraint = qewton.constraints.PINNConstraint(pde_residual)

pde_graph = qewton.PINNPipeline(
    sampler=inner_sampler,
    constraint=pde_constraint,
    models=[model]
)

In [7]:
def dirichlet_residual(u : U):
    return u

dirichlet_constraint = qewton.constraints.PINNConstraint(
    dirichlet_residual, 
    weight=10.0, 
    name="DirichletConstraint"
)
dirichlet_graph = qewton.PINNPipeline(gamma_D_sampler, [model], dirichlet_constraint)

In [8]:
def neumann_residual(u: U, x: X, n: N):
    return u.normal_derivative(n, x)

neuman_constraint = qewton.constraints.PINNConstraint(
    neumann_residual, 
    weight=10.0, 
    name="NeumannConstraint"
)
neumann_graph = qewton.PINNPipeline(gamma_N_sampler, [model], neuman_constraint)

In [9]:
adam_phase = qewton.optim.OptimizationPhase(
    optimizer=qewton.optim.Adam(),
    lr=0.001,
    max_iterations=2500
)

lbfgs_phase = qewton.optim.OptimizationPhase(
    optimizer=qewton.optim.LBFGS(),
    lr=1.0,
    max_iterations=100,
    optimizer_args={"max_eval": 5}
)

trainer = qewton.optim.GraphBasedTrainer(
    optimization_phases=[adam_phase, lbfgs_phase],
    graphs=[pde_graph, dirichlet_graph, neumann_graph],
    training_objectives=[pde_constraint, dirichlet_constraint, neuman_constraint],
    device=qewton.cuda(0)
)

trainer.run()

/home/tomfre/miniconda3/envs/PIONEER/lib/python3.10/site-packages/torch/cuda/__init__.py:716: UserWarning:

Can't initialize NVML

Optimization Phase 2: 100%|██████████| 100/100 [00:16<00:00,  6.22it/s, loss=0.000753]


In [11]:
model.to(qewton.cpu)
solution_data = qewton.DEFAULT_DL_BACKEND.load_data(path + "solution.pt")
reference_config = qewton.DataConfiguration(
    qewton.GeometryAxes(geometry=omega), qewton.FeatureAxes(U)
)

layout = pde_graph.visualize(model.output_ports[0], 
                             reference=solution_data, 
                             reference_config=reference_config)
qewton.viz.Figure(layout).show()